# ebook2audiobook - Android APK build (Google Colab)

Buildozer only runs on Linux, so the APK for the thin Android client
(`android_client/`) is built here on a free Colab machine.

## How to use
1. Runtime type: plain **CPU** is enough (no GPU needed).
2. Run the two cells top to bottom. The first build takes ~30-50 min
   (Android SDK/NDK download); repeat builds are much faster.
3. The finished `ebook2audiobook-*.apk` downloads automatically.
4. On the phone (Android 10/11): allow installing from unknown sources,
   open the APK, done.

## Updates
The APK is signed with a **permanent keystore** that lives on your Google
Drive (shared as a link; cell 2 downloads it automatically via `gdown` -
it is **not** stored in git for security). Because every build shares the
same signature, installing a newer APK over an older one is a normal
**update** - no need to uninstall and reinstall. Keep that keystore
**secret** and never lose it: if it is replaced, phones will refuse an
in-place update. Bump `version` / `version.code` in
`android_client/buildozer.spec` for each release so Android sees a higher
version.

If the build fails, cell (2) prints the error lines from `build.log`
automatically - copy them when reporting a problem.

In [ ]:
#@title (1) Get the client sources & install buildozer
REPO = 'Tarkas/Book-to-audiobook'  #@param {type:"string"}
BRANCH = 'master'  #@param {type:"string"}

import os, subprocess, sys

cmds = [
    'sudo apt-get update -qq',
    # autotools/gettext family is needed to build libffi & friends
    'sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf automake autopoint '
    'libtool libtool-bin libltdl-dev gettext patch pkg-config ccache '
    'zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev',
    # cython 0.29.37 is the newest 0.29.x and supports Python 3.12 (current Colab)
    'pip install -q buildozer cython==0.29.37',
]
for c in cmds:
    print('\n$', c)
    r = subprocess.run(c, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'Setup command failed (exit {r.returncode}): {c}')

if not os.path.isdir('ebook2audiobook'):
    r = subprocess.run(
        f'git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git ebook2audiobook',
        shell=True)
    if r.returncode != 0:
        raise SystemExit('git clone failed - check REPO/BRANCH above')
else:
    # Reused runtime: fetch the requested branch and hard-switch to it, so a
    # BRANCH change (e.g. main -> master) actually takes effect. `git pull`
    # alone would stay on the previously checked-out branch (usually main)
    # and would silently build the WRONG code. The compiled p4a recipes in
    # .buildozer are cached separately and are NOT affected by the branch
    # switch, so a re-run still picks up fixes without a full rebuild.
    r = subprocess.run(
        'git -C ebook2audiobook fetch --depth 1 origin ' + BRANCH, shell=True)
    if r.returncode != 0:
        print('git fetch failed - consider deleting the runtime and rerunning.')
    else:
        r = subprocess.run(
            'git -C ebook2audiobook checkout -B ' + BRANCH + ' origin/' + BRANCH,
            shell=True)
        if r.returncode != 0:
            print('git checkout failed - consider deleting the runtime and rerunning.')

%cd ebook2audiobook/android_client
print('\nSetup done. Python:', sys.version.split()[0])

In [ ]:
#@title (2) Build the APK & download
import glob, os, re, subprocess, sys, time, shutil

# Gradle / Android toolchain needs Java 17 (Colab default may be 11)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# =====================================================
# 1. Download keystore from Google Drive
# =====================================================
KEY_FILE_ID = '19JtV3D33tPmKSI3BKYvEBC4fEancH2Bk'

if not os.path.exists('ebook2audiobook-release.keystore'):
    print('No local keystore - downloading it from Google Drive...')
    rc = subprocess.run(
        'gdown --id ' + KEY_FILE_ID + ' -O ebook2audiobook-release.keystore',
        shell=True
    ).returncode
    if rc != 0 or not os.path.exists('ebook2audiobook-release.keystore'):
        raise SystemExit(
            'Failed to download keystore from Google Drive (file id ' +
            KEY_FILE_ID + ').\n'
            'Make sure the file is shared as "Anyone with the link".'
        )
    os.chmod('ebook2audiobook-release.keystore', 0o600)
else:
    print('Using existing keystore.')

# =====================================================
# 2. Build the APK with buildozer (release)
# =====================================================
print('\n=== Starting buildozer ===')
print('This takes ~30-50 min on first run, much faster later.\n')

# Set environment variables that p4a understands
os.environ['P4A_RELEASE_KEYSTORE'] = os.path.abspath('ebook2audiobook-release.keystore')
os.environ['P4A_RELEASE_KEYALIAS'] = 'ebook2audiobook'
os.environ['P4A_RELEASE_KEYSTORE_PASSWD'] = 'ebook2audiobook2026'
os.environ['P4A_RELEASE_KEYALIAS_PASSWD'] = 'ebook2audiobook2026'

# Run buildozer
with open('build.log', 'w', errors='replace') as log:
    proc = subprocess.Popen(
        'yes | buildozer -v android release',
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors='replace'
    )
    for line in proc.stdout:
        log.write(line)
        # Show progress lines
        if re.search(r'(-> running|# Prepar|# Build|# Download|# Install|# Unpack|# Compil|BUILD SUCCESSFUL|BUILD FAILED|FAILED|sign|Signing)', line, re.I):
            print(line, end='')
    proc.wait()

print('\nbuildozer exit code:', proc.returncode)

# =====================================================
# 3. Find the APK
# =====================================================
apks = sorted(glob.glob('bin/*.apk'))
print('APK files found:', apks if apks else 'NONE')

if not apks:
    print('\n--- No APK found! Build probably failed. ---')
    with open('build.log', errors='replace') as f:
        lines = f.read().splitlines()
    print('\n'.join(lines[-80:]))
    raise SystemExit('No APK produced.')

apk = apks[-1]
print('Raw APK:', apk)

# =====================================================
# 4. Find apksigner with fallback to jarsigner
# =====================================================
def find_apksigner():
    # 1. Check standard Android SDK location
    sdk = os.path.expanduser('~/.buildozer/android/platform/android-sdk')
    build_tools = os.path.join(sdk, 'build-tools')
    if os.path.exists(build_tools):
        versions = sorted([d for d in os.listdir(build_tools) if os.path.isdir(os.path.join(build_tools, d))])
        if versions:
            latest = versions[-1]
            candidate = os.path.join(build_tools, latest, 'apksigner')
            if os.path.exists(candidate):
                return candidate
    
    # 2. Check in PATH
    which = shutil.which('apksigner')
    if which:
        return which
    
    # 3. Try to install via sdkmanager
    sdkmanager = os.path.expanduser('~/.buildozer/android/platform/android-sdk/cmdline-tools/latest/bin/sdkmanager')
    if os.path.exists(sdkmanager):
        print('Installing build-tools via sdkmanager...')
        subprocess.run([sdkmanager, '--install', 'build-tools;33.0.0'], capture_output=True)
        # Try again
        build_tools = os.path.join(sdk, 'build-tools')
        if os.path.exists(build_tools):
            versions = sorted([d for d in os.listdir(build_tools) if os.path.isdir(os.path.join(build_tools, d))])
            if versions:
                latest = versions[-1]
                candidate = os.path.join(build_tools, latest, 'apksigner')
                if os.path.exists(candidate):
                    return candidate
    
    return None

apksigner_cmd = find_apksigner()
keystore = os.path.abspath('ebook2audiobook-release.keystore')

if not os.path.exists(keystore):
    raise SystemExit('Keystore missing: ' + keystore)

# Build signed APK name
signed_apk = apk.replace('-unsigned', '').replace('.apk', '-signed.apk')
if signed_apk == apk:
    signed_apk = apk.replace('.apk', '-signed.apk')

# =====================================================
# 5. Sign the APK (try apksigner first, fallback to jarsigner)
# =====================================================
if apksigner_cmd:
    print('✅ Found apksigner at:', apksigner_cmd)
    
    # Sign with apksigner
    cmd = [
        apksigner_cmd, 'sign',
        '--ks', keystore,
        '--ks-key-alias', 'ebook2audiobook',
        '--ks-pass', 'pass:ebook2audiobook2026',
        '--key-pass', 'pass:ebook2audiobook2026',
        '--out', signed_apk,
        apk,
    ]

    print('\n=== Signing APK with apksigner ===')
    print('Command:', ' '.join(cmd))

    rc = subprocess.run(cmd, text=True, capture_output=True)
    if rc.returncode != 0:
        print('STDERR:', rc.stderr)
        print('STDOUT:', rc.stdout)
        print('\n⚠️ apksigner failed, falling back to jarsigner...')
        apksigner_cmd = None  # Force fallback
    else:
        print('✅ Signed with apksigner')

# Fallback to jarsigner
if not apksigner_cmd:
    print('\n=== Using jarsigner fallback ===')
    
    # Step 1: Sign with jarsigner
    cmd = [
        'jarsigner', '-verbose',
        '-sigalg', 'SHA1withRSA',
        '-digestalg', 'SHA1',
        '-keystore', keystore,
        '-storepass', 'ebook2audiobook2026',
        '-keypass', 'ebook2audiobook2026',
        apk,
        'ebook2audiobook'
    ]
    print('Signing with jarsigner...')
    print('Command:', ' '.join(cmd))
    rc = subprocess.run(cmd, text=True, capture_output=True)
    if rc.returncode != 0:
        print('STDERR:', rc.stderr)
        print('STDOUT:', rc.stdout)
        raise SystemExit('jarsigner failed')
    print('✅ Signed with jarsigner')
    
    # Step 2: zipalign (required for Android)
    zipalign = shutil.which('zipalign')
    if zipalign:
        print('Using zipalign from:', zipalign)
        cmd = [zipalign, '-v', '-p', '4', apk, signed_apk]
        subprocess.run(cmd, capture_output=True)
        print('✅ Zipaligned')
    else:
        # If zipalign not found, just copy
        import shutil as sh
        sh.copy2(apk, signed_apk)
        print('⚠️ zipalign not found - skipping (APK might still work)')

# =====================================================
# 6. Verify signature
# =====================================================
print('\n=== Verifying signature ===')
if apksigner_cmd:
    verify_cmd = [apksigner_cmd, 'verify', '--verbose', signed_apk]
    vr = subprocess.run(verify_cmd, text=True, capture_output=True)
    print(vr.stdout if vr.stdout else vr.stderr)
    if vr.returncode != 0:
        print('⚠️ WARNING: Signature verification failed?')
    else:
        print('✅ Signature verified OK')
else:
    # Verify with jarsigner
    verify_cmd = ['jarsigner', '-verify', '-verbose', '-certs', signed_apk]
    vr = subprocess.run(verify_cmd, text=True, capture_output=True)
    print(vr.stdout if vr.stdout else vr.stderr)
    if vr.returncode != 0:
        print('⚠️ WARNING: Signature verification failed?')
    else:
        print('✅ Signature verified OK')

# =====================================================
# 7. Download the signed APK
# =====================================================
print('\n✅ SIGNED APK:', signed_apk)
print('📁 Unsigned APK is still at:', apk)

try:
    from google.colab import files
    print('\n=== Downloading signed APK ===')
    files.download(signed_apk)
    print('\n✅ Done! The signed APK has been downloaded.')
except ImportError:
    print('\nNot running in Colab? Grab the file from:', signed_apk)
except Exception as e:
    print('\nDownload failed:', e)
    print('Grab the file manually from:', signed_apk)

In [ ]:
#@title (3) Diagnostics - real failures only + tail of build.log
import glob, re

apks = sorted(glob.glob('bin/*.apk'))
print('APK files:', apks if apks else 'NONE')

lines = open('build.log', errors='replace').read().splitlines()

# Real failure markers only. p4a's "Trying first build ... this is expected
# to fail" passes produce harmless clang/ccache errors - do not report them.
real = [i for i, l in enumerate(lines)
        if re.search(r'# Command failed|BUILD FAILED|Aborted!|buildozer.*[Ee]rror', l)]
if real:
    for i in real:
        lo, hi = max(0, i - 40), min(len(lines), i + 10)
        print('=' * 70)
        print('\n'.join(lines[lo:hi]))
    print('=' * 70)
    print('Copy everything above when reporting the problem.')
else:
    print('No real failure markers found.')

print('\n--- last 60 lines of build.log ---')
print('\n'.join(lines[-60:]))

# Save the same diagnostics to a file and download it: no copying needed.
with open('build_report.txt', 'w', errors='replace') as rep:
    for i in real[-10:]:
        rep.write('=' * 70 + '\n')
        rep.write('\n'.join(lines[max(0, i - 40):i + 10]) + '\n')
    rep.write('=' * 70 + '\n--- last 200 lines ---\n')
    rep.write('\n'.join(lines[-200:]) + '\n')
try:
    from google.colab import files
    files.download('build_report.txt')
    print('Downloaded build_report.txt - send this file.')
except Exception as e:
    print('Auto-download failed:', e, '- grab build_report.txt from the Files panel.')